In [1]:
pip list

Package                           Version
--------------------------------- ------------------
aiobotocore                       3.7.0
aiodns                            3.6.1
aiohappyeyeballs                  2.6.2
aiohttp                           3.14.1
aioitertools                      0.12.0
aiosignal                         1.4.0
alabaster                         1.0.0
altair                            6.0.0
anaconda-anon-usage               0.8.1
anaconda-auth                     0.15.1
anaconda-catalogs                 0.2.0
anaconda-channel-guide            0.1.1
anaconda-cli-base                 0.9.1
anaconda-client                   1.14.1
anaconda-navigator                2.7.1
anaconda-project                  0.13.2
annotated-doc                     0.0.4
annotated-types                   0.7.0
anyio                             4.12.1
archspec                          0.2.6
argon2-cffi                       25.1.0
argon2-cffi-bindings              25.1.0
arrow            

In [4]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np
import time
import random

BASE_URL = "https://books.toscrape.com/"
print("Importado correctamente")

Importado correctamente


In [5]:
response = requests.get(BASE_URL, timeout=10)
print("Codigo de estado: ", response.status_code)
print("Longitud del html: ", len(response.text))

      
print(response.text[:500])

Codigo de estado:  200
Longitud del html:  51294
<!DOCTYPE html>
<!--[if lt IE 7]>      <html lang="en-us" class="no-js lt-ie9 lt-ie8 lt-ie7"> <![endif]-->
<!--[if IE 7]>         <html lang="en-us" class="no-js lt-ie9 lt-ie8"> <![endif]-->
<!--[if IE 8]>         <html lang="en-us" class="no-js lt-ie9"> <![endif]-->
<!--[if gt IE 8]><!--> <html lang="en-us" class="no-js"> <!--<![endif]-->
    <head>
        <title>
    All products | Books to Scrape - Sandbox
</title>

        <meta http-equiv="content-type" content="text/html; charset=UTF-8" /


In [6]:
soup = BeautifulSoup(response.text, "html.parser")

#Buscamos todos los elementos <article class="product_pod">

libros_html = soup.select("article.product_pod")
print(f"Se encontraron {len(libros_html)} libros en esta pagina")

#Veamos el primero
print(libros_html[0].prettify()[:600])

Se encontraron 20 libros en esta pagina
<article class="product_pod">
 <div class="image_container">
  <a href="catalogue/a-light-in-the-attic_1000/index.html">
   <img alt="A Light in the Attic" class="thumbnail" src="media/cache/2c/da/2cdad67c44b002e7ead0cc35693c0e8b.jpg"/>
  </a>
 </div>
 <p class="star-rating Three">
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
 </p>
 <h3>
  <a href="catalogue/a-light-in-the-attic_1000/index.html" title="A Light in the Attic">
   A Light in the ...
  </a>
 </h3>
 <div class="product_pric


In [10]:
def extraer_rating(clases_css):
    """El rating viene como una clase CSS: 'star-rating Three'->3"""
    mapa = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}
    for clase in clases_css:
        if clase in mapa:
            return mapa[clase]
        return None

libros = []
for item in libros_html:
    titulo = item.h3.a["title"]
    precio_texto = item.select_one("p.price_color").text.strip()
    precio = float(precio_texto.replace("£", "").replace("Â", ""))
    disponibilidad = item.select_one("p.instock.availability").text.strip()
    rating_clases = item.select_one("p.star-rating")["class"]
    rating = extraer_rating(rating_clases)

    enlace_relativo = item.h3.a["href"]

    libros.append({
        "titulo": titulo,
        "precio_gbp": precio,
        "disponibilidad": disponibilidad,
        "rating": rating,
        "enlace": BASE_URL + enlace_relativo
    })

df_pagina1 = pd.DataFrame(libros)
df_pagina1.head(10)
    

,titulo,precio_gbp,disponibilidad,rating,enlace
0,A Light in the Attic,51.77,In stock,None,https://books.toscrape.com/catalogue/a-light-i...
1,Tipping the Velvet,53.74,In stock,None,https://books.toscrape.com/catalogue/tipping-t...
2,Soumission,50.10,In stock,None,https://books.toscrape.com/catalogue/soumissio...
3,Sharp Objects,47.82,In stock,None,https://books.toscrape.com/catalogue/sharp-obj...
4,Sapiens: A Brief History of Humankind,54.23,In stock,None,https://books.toscrape.com/catalogue/sapiens-a...
5,The Requiem Red,22.65,In stock,None,https://books.toscrape.com/catalogue/the-requi...
6,The Dirty Little Secrets of Getting Your Dream...,33.34,In stock,None,https://books.toscrape.com/catalogue/the-dirty...
7,The Coming Woman: A Novel Based on the Life of...,17.93,In stock,None,https://books.toscrape.com/catalogue/the-comin...
8,The Boys in the Boat: Nine Americans and Their...,22.60,In stock,None,https://books.toscrape.com/catalogue/the-boys-...
9,The Black Maria,52.15,In stock,None,https://books.toscrape.com/catalogue/the-black...


In [11]:
df_pagina1.to_csv("Libros_toscrape.pag1.cvs", index=False, encoding="utf-8-sig")
df_pagina1.to_excel("Libros_toscrape.pag1.xlsx", index=False)
